# Football Analytics Package — Demo Notebook

This notebook demonstrates the full capabilities of the `football-analytics` package.

We use two data sources:
- **StatsBomb open data** — player-level event data, xG, passes, defensive actions
- **football-data.org** — standings, fixtures, match results

### Sections
1. Setup and imports
2. Exploring available StatsBomb data
3. Match level analysis
4. Player comparison — key players across La Liga seasons
5. Season level analysis and comparison

### 1. Setup and imports 

In [5]:
# standard library
import warnings

# data
import pandas as pd

warnings.filterwarnings("ignore")

# our package — data layer
from football_analytics.data import FootballDataClient, StatsBombClient  # noqa: E402, I001

# our package — analytics layer
from football_analytics.analytics import (  # noqa: E402
    # form
    get_recent_form,
    get_points_per_game,
    get_home_away_split,
    # xG
    get_match_xg_summary,
    get_player_xg_ranking,
    get_xg_overperformance,
    # standings
    get_clean_standings,
    get_expected_vs_actual,
    # player
    get_top_performers,
    add_per_90_columns,
    per_90,
)

print("imports OK")

imports OK


In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# Initialise clients 

# StatsBomb requires no API key 
sb_client = StatsBombClient()

# FootballDataClient reads the key from .env aautomatically 
fd_client = FootballDataClient()

print("Clients initialised OK")

Clients initialised OK


### 2. Exploring available StatsBomb 

First we can look at what data from which competitions are contained in the open package of StatsBomb.

In [8]:
competitions = sb_client.get_competitions()
competitions

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,NaN,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,NaN,2024-02-13T02:35:28.134882
...,...,...,...,...,...,...,...,...,...,...,...,...
75,35,75,Europe,UEFA Europa League,male,False,False,1988/1989,2026-04-11T12:48:10.012987,2021-06-13T16:17:31.694,NaN,2026-04-11T12:48:10.012987
76,53,315,Europe,UEFA Women's Euro,female,False,True,2025,2026-04-27T22:02:42.690507,2026-04-27T22:03:28.087062,2026-04-27T22:03:28.087062,2026-04-27T22:02:42.690507
77,53,106,Europe,UEFA Women's Euro,female,False,True,2022,2026-05-05T03:03:04.199896,2026-05-05T03:05:32.480837,2026-05-05T03:05:32.480837,2026-05-05T03:03:04.199896
78,72,107,International,Women's World Cup,female,False,True,2023,2026-05-03T13:51:31.021141,2026-05-03T13:55:52.303219,2026-05-03T13:55:52.303219,2026-05-03T13:51:31.021141


In [9]:
# Let's see all the available competitions  
competitions.competition_name.unique()

<StringArray>
[          '1. Bundesliga',  'African Cup of Nations',
        'Champions League',            'Copa America',
            'Copa del Rey', 'FA Women's Super League',
      'FIFA U20 World Cup',          'FIFA World Cup',
       'Frauen Bundesliga',     'Indian Super league',
                 'La Liga',                  'Liga F',
        'Liga Profesional',                 'Ligue 1',
     'Major League Soccer',   'North American League',
                    'NWSL',          'Premier League',
                 'Serie A',           'Serie A Women',
               'UEFA Euro',      'UEFA Europa League',
       'UEFA Women's Euro',       'Women's World Cup']
Length: 24, dtype: str

In [10]:
# As a domain knowledge, we're aware that open StatsBomb data contains
# a lot of La Liga data.  Below is shown all the available seasons for LaLiga
 
bundesliga = competitions[
    competitions["competition_name"].str.contains("liga", case=False)
]
print(bundesliga[["competition_id", "season_id", "competition_name", "season_name"]])


    competition_id  season_id   competition_name season_name
0                9        281      1. Bundesliga   2023/2024
1                9         27      1. Bundesliga   2015/2016
38             135        281  Frauen Bundesliga   2023/2024
40              11         90            La Liga   2020/2021
41              11         42            La Liga   2019/2020
42              11          4            La Liga   2018/2019
43              11          1            La Liga   2017/2018
44              11          2            La Liga   2016/2017
45              11         27            La Liga   2015/2016
46              11         26            La Liga   2014/2015
47              11         25            La Liga   2013/2014
48              11         24            La Liga   2012/2013
49              11         23            La Liga   2011/2012
50              11         22            La Liga   2010/2011
51              11         21            La Liga   2009/2010
52              11      

### 3. Match Level Analysis 

We start our data exploration from the smallest possible unit of comparison for our package, and that is a single match. Here we demonstrate what match-level analysis can be done with the package. 

We use a match from La Liga season 2015/16 (competition_id=11, season_id=27).

In [11]:
# retrieving all the matches in the given season 
COMPETITION_ID = 11 
SEASON_ID = 27 # 2015/16 

laliga_matches_15 = sb_client.get_matches(competition_id=COMPETITION_ID,
                                          season_id=SEASON_ID)

print ( f"Total matches in 2015/16: {len(laliga_matches_15)}")
laliga_matches_15[["match_id", "home_team", "away_team", "home_score", "away_score"]].head(10)

Total matches in 2015/16: 380


,match_id,home_team,away_team,home_score,away_score
0,3825739,Real Madrid,Sporting Gijón,5,1
1,3825848,Levante UD,Eibar,2,2
2,3825895,Las Palmas,Sevilla,2,0
3,3825894,RC Deportivo La Coruña,Getafe,0,2
4,3825855,Málaga,Levante UD,3,1
5,3825908,Espanyol,Eibar,4,2
6,3825883,Málaga,Las Palmas,4,1
7,3825900,Sporting Gijón,Villarreal,2,0
8,3825902,Rayo Vallecano,Levante UD,3,1
9,3825876,Real Betis,Getafe,2,1


In [12]:
# picking one match to explore from Barcelona matches 

barca_matches = sb_client.get_matches(competition_id=COMPETITION_ID,
                                      season_id=SEASON_ID,
                                      team="Barcelona")

print(f"Barcelona matches available: expected -> 38 retrieved -> {len(barca_matches)}")
barca_matches[["match_id", "home_team", "away_team", "home_score", "away_score"]].head(10)

Barcelona matches available: expected -> 38 retrieved -> 38


,match_id,home_team,away_team,home_score,away_score
173,3825660,Barcelona,Villarreal,3,0
186,3825637,Barcelona,Eibar,3,1
288,3825645,Getafe,Barcelona,0,2
300,3825627,Barcelona,Rayo Vallecano,5,2
306,3825617,Sevilla,Barcelona,2,1
347,266498,Barcelona,Getafe,6,0
348,266986,Real Betis,Barcelona,0,2
349,267533,Barcelona,Real Madrid,1,2
350,266310,RC Deportivo La Coruña,Barcelona,0,8
351,267576,Barcelona,Atlético Madrid,2,1


#### 3.1 xG Summary for a Single Match 

`get_shots()` returns the raw shot daa for a match
`get_match_xg_summary()` takes raw shot data and returns a team-level summary of total shots, shots on target, xG and goals. 
xg_difference tells us how much a team over or underperformed their chances. 

In [13]:
# use the first Barcelona match
MATCH_ID = barca_matches.iloc[0]["match_id"]
home = barca_matches.iloc[0]["home_team"]
away = barca_matches.iloc[0]["away_team"]
home_score = barca_matches.iloc[0]["home_score"]
away_score = barca_matches.iloc[0]["away_score"]

print(f"Match: {home} {home_score} - {away_score} {away}\n")

shots = sb_client.get_shots(match_id=MATCH_ID)
xg_summary = get_match_xg_summary(shots)
xg_summary

Match: Barcelona 3 - 0 Villarreal



,team,total_shots,shots_on_target,total_xg,goals,xg_difference
0,Barcelona,20,9,2.870,3,0.130
1,Villarreal,5,2,0.326,0,-0.326


#### 3.2 Player Shooting Stats for a Single Match 

`get_player_shooting_match()` retruns player level xG data for a single match with data on shots taken, shots on target, goals scored and xG generated

In [14]:
shooting_stats = sb_client.get_player_shooting_match(match_id=MATCH_ID)
shooting_stats.sort_values('total_xg', ascending=False)


,player,team,shots,shots_on_target,goals,total_xg,xg_per_shot
5,Luis Alberto Suárez Díaz,Barcelona,7,2,1,1.106,0.158
0,Andrés Iniesta Luján,Barcelona,2,1,0,0.570,0.285
8,Neymar da Silva Santos Junior,Barcelona,4,3,2,0.494,0.124
2,Daniel Alves da Silva,Barcelona,2,1,0,0.231,0.116
4,Jérémy Mathieu,Barcelona,2,2,0,0.175,0.088
7,Munir El Haddadi Mohamed,Barcelona,1,0,0,0.152,0.152
11,Sergi Roberto Carnicer,Barcelona,2,0,0,0.141,0.070
10,Samuel Castillejo Azuaga,Villarreal,1,1,0,0.126,0.126
1,Bruno Soriano Llido,Villarreal,1,0,0,0.060,0.060
3,Denis Suárez Fernández,Villarreal,1,0,0,0.049,0.049
